# Phase 3 — Feature Quality & Pareto

**Objective:** Every feature the model sees must have a well-behaved distribution.

Loads `features.parquet` from Notebook 2, plots distributions, computes skewness,
applies winsorising and log1p where needed, and saves the transformed table as
`features_transformed.parquet`.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

USER_COL = "ACCOUNT_ID"

features_df = pl.read_parquet("exp_files/features.parquet")
feature_columns = features_df.columns[1:]
print(f"Loaded features_df — shape: {features_df.shape}")
feature_columns

Loaded features_df — shape: (850000, 37)


['balance_std_dev_last30D',
 'balance_std_dev_last14D',
 'balance_std_dev_last7D',
 'balance_trend',
 'bill_pay_count_last_30D',
 'bill_pay_count_last_14D',
 'bill_pay_count_last_7D',
 'merchant_count_last_30D',
 'merchant_count_last_14D',
 'merchant_count_last_7D',
 'p2p_count_last_30D',
 'p2p_count_last_14D',
 'p2p_count_last_7D',
 'cash_in_amount_last_30D',
 'cash_in_count_last_14D',
 'cash_in_count_last_7D',
 'cash_out_count_last_30D',
 'cash_out_count_last_14D',
 'cash_out_count_last_7D',
 'bill_pay_ratio',
 'unique_trx_types_used',
 'total_trx_amount',
 'frequency',
 'trx_amount_last_30D',
 'freq_last_30D',
 'recency',
 'average_gap_days',
 'trx_trend',
 'consistency_metric',
 'p2p_ratio_90d',
 'importance',
 'freq_last_14D',
 'freq_last_7D',
 'velocity_last_14d',
 'velocity_last_7d',
 'inactivity_multiplier']

## Distribution plotting helper

In [2]:
def show_dist_plots(feature_column, features_df, col_skew):
    print("\nGenerating distribution plots...")
    sns.set_theme(style="whitegrid")
    for col in feature_column:
        fig, ax = plt.subplots(figsize=(7,4))
        data_array = features_df[col].to_numpy()

        sns.histplot(data_array, kde=True, ax=ax, color="darkcyan", bins=30)
        ax.set_title(f"Distribution of {col} (Skewness: {col_skew[col][0]:.2f})", fontsize=12, fontweight='bold')
        ax.set_xlabel(col, fontsize=10)
        ax.set_ylabel("Count", fontsize=10)
        plt.show()

## Step 1 — Measure raw skewness for every feature

In [3]:
print("\n=== FEATURE SKEWNESS VALUES (before transform) ===")
# Polars computes skewness across all features in parallel
skew_df = features_df.select([
    pl.col(col).skew().alias(col) for col in feature_columns
])
prev_skew = skew_df.to_numpy()
skew_df


=== FEATURE SKEWNESS VALUES (before transform) ===


balance_std_dev_last30D,balance_std_dev_last14D,balance_std_dev_last7D,balance_trend,bill_pay_count_last_30D,bill_pay_count_last_14D,bill_pay_count_last_7D,merchant_count_last_30D,merchant_count_last_14D,merchant_count_last_7D,p2p_count_last_30D,p2p_count_last_14D,p2p_count_last_7D,cash_in_amount_last_30D,cash_in_count_last_14D,cash_in_count_last_7D,cash_out_count_last_30D,cash_out_count_last_14D,cash_out_count_last_7D,bill_pay_ratio,unique_trx_types_used,total_trx_amount,frequency,trx_amount_last_30D,freq_last_30D,recency,average_gap_days,trx_trend,consistency_metric,p2p_ratio_90d,importance,freq_last_14D,freq_last_7D,velocity_last_14d,velocity_last_7d,inactivity_multiplier
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
11.306718,10.165154,9.69611,0.965677,4.071993,4.08612,4.11036,3.306721,3.340968,3.394931,3.687925,3.714648,3.767341,8.564679,4.463493,4.469714,5.782011,5.787764,5.827561,1.356623,0.020186,2.896781,1.583829,2.983752,1.576636,3.213245,3.492779,3.265818,-0.806345,1.329647,2.900431,1.593506,1.629857,-0.195631,0.622975,12.308008


In [4]:
# show_dist_plots(feature_columns, features_df, skew_df)

## Step 2 — Winsorising

These features have either heavy outliers or are sensitive to extreme values where
clipping at a percentile (rather than log-transforming) is more appropriate —
e.g. ratios and counts that are bounded in a meaningful range.

In [5]:
winrorise_columns = {
    # 'balance_std_dev_last30D': [0.0, 0.98],
    'balance_std_dev_last14D': [0.0, 0.98],
    'balance_std_dev_last7D': [0.0, 0.98],
    'bill_pay_count_last_30D': [0.0, 0.98],
    'bill_pay_count_last_14D': [0.0, 0.97],
    # 'bill_pay_count_last_7D': [0.0, 0.99],
    # 'merchant_count_last_30D': [0.0, 0.99],
    # 'merchant_count_last_14D': [0.0, 0.99],
    # 'merchant_count_last_7D': [0.0, 0.99],
    # 'p2p_count_last_30D': [0.0, 0.99],
    # 'p2p_count_last_14D': [0.0, 0.99],
    # 'p2p_count_last_7D': [0.0, 0.99],
    # 'cash_in_count_last_30D': [0.0, 0.98],
    # 'cash_in_count_last_14D': [0.0, 0.98],
    # 'cash_in_count_last_7D': [0.0, 0.98],
    # 'cash_out_count_last_30D': [0.0, 0.98],
    # 'cash_out_count_last_14D': [0.0, 0.98],
    # 'cash_out_count_last_7D': [0.0, 0.98],

    'inactivity_multiplier': [0.0, 0.99],
    'freq_last_30D': [0.05, 0.95],
    'recency': [0.0, 0.93],
    'average_gap_days': [0.0, 0.95],
    'importance': [0.0, 0.95],
    'trx_trend': [0.0, 0.99]
}
log1p_columns = []

In [6]:
winsorising = []

for col in winrorise_columns:
    q_low = features_df.select(pl.col(col).quantile(winrorise_columns[col][0])).item()
    q_high = features_df.select(pl.col(col).quantile(winrorise_columns[col][1])).item()
    winsorising.append(
        pl.col(col)
        .clip(lower_bound=q_low, upper_bound=q_high)
    )

transformed_df = features_df.with_columns(winsorising)

## Step 3 — log1p transform

Applied across all numeric feature columns (after winsorising) to compress
remaining right-skew. Safe for zeros since `log1p(0) = 0`.

In [7]:
log1p_cols = feature_columns
logify = []
for col in log1p_cols:
    logify.append(
        pl.col(col)
        .log1p()
    )

transformed_df2 = transformed_df.with_columns(logify)
transformed_df2

ACCOUNT_ID,balance_std_dev_last30D,balance_std_dev_last14D,balance_std_dev_last7D,balance_trend,bill_pay_count_last_30D,bill_pay_count_last_14D,bill_pay_count_last_7D,merchant_count_last_30D,merchant_count_last_14D,merchant_count_last_7D,p2p_count_last_30D,p2p_count_last_14D,p2p_count_last_7D,cash_in_amount_last_30D,cash_in_count_last_14D,cash_in_count_last_7D,cash_out_count_last_30D,cash_out_count_last_14D,cash_out_count_last_7D,bill_pay_ratio,unique_trx_types_used,total_trx_amount,frequency,trx_amount_last_30D,freq_last_30D,recency,average_gap_days,trx_trend,consistency_metric,p2p_ratio_90d,importance,freq_last_14D,freq_last_7D,velocity_last_14d,velocity_last_7d,inactivity_multiplier
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""CUST00066387""",1.270641,1.461517,1.600682,0.028485,2.197225,1.609438,1.098612,1.94591,1.098612,0.693147,2.995732,2.397895,1.94591,0.0,0.0,0.0,2.70805,2.079442,1.386294,0.083401,1.609438,13.51753,4.820282,12.117587,3.850148,0.0,0.277383,0.868089,1.087866,0.492936,0.718158,3.178054,2.564949,0.170151,0.092373,0.0
"""CUST00066530""",2.026756,0.643358,0.0,0.000684,1.386294,1.098612,1.098612,1.098612,1.098612,1.098612,1.609438,1.386294,0.0,0.0,0.0,0.0,0.693147,0.693147,0.693147,0.135833,1.609438,11.614619,3.663562,10.514019,2.397895,0.693147,1.014055,0.523248,0.756678,0.539605,0.14556,2.197225,1.791759,0.186586,0.120628,0.430856
"""CUST00130604""",1.485887,0.0,0.0,0.003872,3.828641,3.044522,2.772589,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.098612,0.693147,0.693147,0.685894,1.386294,13.470791,4.962845,11.986698,3.850148,0.0,0.210295,0.662688,0.96527,0.0,0.694485,3.295837,2.833213,0.167054,0.10606,0.0
"""CUST00131197""",13.029786,10.676394,10.279175,1.011016,1.098612,0.693147,0.693147,0.0,0.0,0.0,1.386294,1.098612,0.693147,14.494881,3.555348,2.70805,3.583519,2.995732,2.564949,0.032759,1.791759,15.643471,5.888878,14.824207,4.691348,0.0,0.022039,0.651553,0.792431,0.060788,2.116019,4.043051,3.367296,0.144208,0.074701,0.0
"""CUST00131592""",8.303442,8.545286,2.336726,0.68587,0.693147,0.693147,0.0,1.791759,1.098612,0.0,0.693147,0.693147,0.693147,8.883282,0.0,0.0,0.0,0.0,0.0,0.057104,1.791759,11.246375,3.496508,9.80424,2.302585,1.94591,1.130361,0.646627,0.758762,0.119188,0.102934,1.609438,0.693147,0.11441,0.029853,1.316751
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""CUST00849218""",0.83069,0.71448,0.927024,0.001468,0.0,0.0,0.0,3.091042,2.397895,1.94591,3.258097,2.639057,2.197225,8.319076,0.693147,0.693147,0.0,0.0,0.0,0.0,1.386294,13.644417,4.859812,12.392256,3.78419,0.0,0.291611,0.693147,0.947468,0.258758,0.785177,3.218876,2.772589,0.170626,0.110001,0.0
"""CUST00070334""",1.639307,1.84856,2.139706,0.014827,1.386294,0.693147,0.0,1.609438,0.693147,0.0,2.197225,0.0,0.0,5.662544,0.0,0.0,0.0,0.0,0.0,0.073557,1.609438,11.698751,3.663562,11.224342,2.772589,2.484907,0.97405,0.76214,0.931429,0.491408,0.15738,1.098612,0.0,0.05001,0.0,1.986582
"""CUST00070408""",0.757723,0.953564,1.128218,0.000993,3.663562,2.70805,2.079442,0.0,0.0,0.0,3.135494,2.564949,2.079442,0.0,0.0,0.0,1.609438,1.386294,1.098612,0.387212,1.609438,13.448724,5.043425,12.449813,4.094345,0.0,0.221836,0.847298,0.95118,0.395088,0.683497,3.401197,2.833213,0.171511,0.098238,0.0


## Step 4 — Measure skewness after transform, compare before/after

In [8]:
skew_df = transformed_df2.select([
    pl.col(col).skew().alias(col) for col in feature_columns
])
after_skew = skew_df.to_numpy()


In [9]:
# show_dist_plots(feature_columns, transformed_df2, skew_df)

In [10]:
skew_table = pl.DataFrame({
    "Features": feature_columns,
    "Skewness(before)": prev_skew.flatten(),
    "Skewness(after)" : after_skew.flatten()
})

with pl.Config(tbl_rows=25):
    display(skew_table)

Features,Skewness(before),Skewness(after)
str,f64,f64
"""balance_std_dev_last30D""",11.306718,0.389913
"""balance_std_dev_last14D""",10.165154,0.616166
"""balance_std_dev_last7D""",9.69611,0.893729
"""balance_trend""",0.965677,0.482776
"""bill_pay_count_last_30D""",4.071993,0.9071
"""bill_pay_count_last_14D""",4.08612,1.129085
"""bill_pay_count_last_7D""",4.11036,1.524384
"""merchant_count_last_30D""",3.306721,0.799927
"""merchant_count_last_14D""",3.340968,1.034146


## Final feature list used for modeling

This explicit list is the contract between this notebook and the next — it should
match the columns actually present in `transformed_df2`.

## Save transformed features for downstream notebooks

> **Note:** if you re-run the full pipeline (rather than re-reading a cached file),
> recompute `transformed_df2` from `final_pipe` directly to avoid double-transforming
> already-saved data.

In [11]:
transformed_df2.write_parquet("exp_files/features_transformed.parquet")
print(f"Saved features_transformed.parquet — shape: {transformed_df2.shape}")

Saved features_transformed.parquet — shape: (850000, 37)


## Next notebook

Continue to **`04_class_imbalance.ipynb`**, which loads `features_transformed.parquet`,
joins labels, and builds the train/validation split.